# 07f — Cross-City Generalization

Not k-fold CV: trains on ALL of one city's data, tests on ALL of the
other city's data. Two directions: Bogor→Warsaw and Warsaw→Bogor.

This is the track that actually answers "does the model generalize
across cities" — `07e` (pooled) answers a different question ("does
more data, drawn from both cities, improve in-distribution performance").
Both matter; they're not substitutes for each other.

**Repeats, not folds:** since there's exactly one train/test partition
per direction (not one per fold), variance here comes from re-running
with different seeds (train/val split + model init), not from resampling
the partition itself. This is a DIFFERENT statistical object than the
repeated-spatial-k-fold scores `07`/`07e` produce — reported as
descriptive mean±std across `N_REPEATS` seeds, NOT run through
Wilcoxon/Nadeau-Bengio (those assume paired, spatially-blocked folds).
Uses the same `eval.yaml` repeats count (3) as a default, matching what
you already use elsewhere — change `N_REPEATS` below if you want more.

**Requires `04b`+`05b`** — reads the unified/pooled index so train and
test share the exact same vocab, and requires the same `svg_kwargs`/
`tvg_kwargs` vocab sizes as `07e`.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml, json
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/eval.yaml") as f:
    eval_cfg = yaml.safe_load(f)

COMBINED_BASE_DIR = Path("/content/drive/MyDrive/crash-dualgraph/data/combined")
PROCESSED_DIR = COMBINED_BASE_DIR / "processed"
OUTPUTS_DIR = COMBINED_BASE_DIR / "outputs"
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_cross_city"
METRICS_DIR = OUTPUTS_DIR / "metrics"
for d in [CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
config = {"batch_size": eval_cfg.get("batch_size") or 32,
          "epoch_cap": eval_cfg.get("epoch_cap") or 150,
          "patience": eval_cfg.get("patience") or 20}
N_REPEATS = eval_cfg.get("repeats") or 3
print(f"Device: {device} | config: {config} | N_REPEATS: {N_REPEATS}")

In [ ]:
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models

index_df = pd.read_parquet(PROCESSED_DIR / "dataset_index.parquet")
dataset = ds.DualGraphDataset(index_df)
print(f"Pooled dataset: {len(dataset)} points")
display(index_df.groupby("city").size().rename("n_points"))

BOGOR_CACHE = Path("/content/drive/MyDrive/crash-dualgraph/data/bogor/interim/osm_cache")
with open(BOGOR_CACHE / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(BOGOR_CACHE / "building_type_vocab.json") as f:
    BUILDING_VOCAB_SIZE = len(json.load(f))

svg_kwargs = dict(hidden_dim=64, heads=4, num_layers=2, dropout=0.35,
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2, cat_embed_dim=2)
tvg_kwargs = dict(hidden_dim=64, heads=4, num_layers=2, dropout=0.35,
                   building_type_vocab=BUILDING_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=8, highway_embed_dim=4)
print(f"highway_vocab={HIGHWAY_VOCAB_SIZE}  building_type_vocab={BUILDING_VOCAB_SIZE}")

In [ ]:
# ── Core loop: one (scenario, depth, ablation, direction) at a time,
# repeated N_REPEATS times with different seeds. Checkpointed exactly
# like run_scenario — skips completed (direction, repeat) entries on rerun.
def run_cross_city(scenario, head_depth, use_ablation, train_city, test_city, n_repeats,
                    val_frac=0.15, verbose=True):
    tag = f"{scenario}_{head_depth}{'_ablation' if use_ablation else ''}_{train_city}to{test_city}"
    results_path = CHECKPOINT_DIR / f"{tag}_results.json"
    results = json.loads(results_path.read_text()) if results_path.exists() else []
    done_seeds = {r["seed"] for r in results}

    train_full_df = index_df[index_df["city"] == train_city]
    test_df = index_df[index_df["city"] == test_city]

    for rep in range(n_repeats):
        seed = 42 + rep
        if seed in done_seeds:
            continue

        val_df = train_full_df.sample(frac=val_frac, random_state=seed)
        train_df = train_full_df.drop(val_df.index)

        def _items(df):
            return [dataset[i] for i in df.index]

        if verbose:
            print(f"  -- {tag} seed={seed} (n_train={len(train_df)}, n_val={len(val_df)}, n_test={len(test_df)}) --")

        history_path = CHECKPOINT_DIR / f"{tag}_seed{seed}_history.json"
        test_metrics, _ = tr.train_one_fold(
            scenario, head_depth, use_ablation, _items(train_df), _items(val_df), _items(test_df),
            svg_kwargs, tvg_kwargs, config, device, verbose=verbose, history_path=history_path,
        )

        results.append({"seed": seed, "n_train": len(train_df), "n_test": len(test_df), **test_metrics})
        results_path.write_text(json.dumps(tr._to_jsonable(results), indent=1))

    return results

### Run both directions, primary scenarios A–E (winning head depth only — set below once 07/07e have picked one; defaults to 'linear' if run standalone)

In [ ]:
# Reuse whichever head depth 07/07e already settled on, if that summary
# exists; otherwise fall back to 'linear' rather than blocking here.
pooled_summary_path = OUTPUTS_DIR / "metrics" / "all_scenarios_summary_pooled.csv"
if pooled_summary_path.exists():
    _agg = pd.read_csv(pooled_summary_path)
    _lin = _agg[_agg["scenario"].str.endswith("_linear")]["pr_auc_mean"].mean()
    _mlp = _agg[_agg["scenario"].str.endswith("_mlp2") & ~_agg["scenario"].str.contains("ablation")]["pr_auc_mean"].mean()
    WINNING_DEPTH = "linear" if _lin >= _mlp else "mlp2"
else:
    WINNING_DEPTH = "linear"
print(f"Using head depth: {WINNING_DEPTH}")

PRIMARY_SCENARIOS = ["A", "B", "C", "D", "E"]
DIRECTIONS = [("bogor", "warsaw"), ("warsaw", "bogor")]
cross_city_results = {}

for scenario in PRIMARY_SCENARIOS:
    for train_city, test_city in DIRECTIONS:
        key = f"{scenario}_{train_city}to{test_city}"
        print(f"\n=== {key} ===")
        cross_city_results[key] = run_cross_city(
            scenario, WINNING_DEPTH, use_ablation=False,
            train_city=train_city, test_city=test_city, n_repeats=N_REPEATS,
        )

In [ ]:
# ── Aggregate (descriptive mean±std across seeds — not a formal test) ────
agg_rows = []
for key, results in cross_city_results.items():
    agg = ev.aggregate_fold_results(results)
    row = {"run": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

cross_city_df = pd.DataFrame(agg_rows)
cross_city_df.to_csv(METRICS_DIR / "cross_city_summary.csv", index=False)
display(cross_city_df)

## Reading this alongside 07 (single-city) and 07e (pooled)

- If cross-city PR-AUC/AUROC is far below each city's OWN k-fold CV
  number (from `07`), that's the generalization gap — the model is
  learning city-specific patterns that don't transfer.
- If pooled (`07e`) beats both single-city baselines (`07`), that's
  evidence more data helps even without cross-city transfer being clean.
- Both can be true at once: pooling can help in-distribution performance
  while cross-city transfer stays weak — worth stating explicitly in the
  writeup rather than treating 'combining helps' as one single verdict.